In [ ]:
import numpy as np
import os
import os.path as op
import pandas as pd
from alternet.splicefactor_evidence import *
from alternet.compare_nets import *

from alternet.gtex_dataloader import *
from alternet.annotation import *





gtex_data_dir = '/data/bionets/datasets/hackathon/data/GTEX'
tissues = ['Blood', 'Brain', 'Adipose Tissue', 'Muscle', 'Blood Vessel','Heart']
T = ['Ovary', 'Uterus', 'Vagina', 'Breast', 'Skin']
tissues = [ 'Salivary Gland', 'Adrenal Gland', 'Thyroid', 'Lung', 'Spleen',
       'Pancreas', 'Esophagus', 'Stomach', 'Colon', 'Small Intestine',
       'Prostate', 'Testis', 'Nerve', 'Pituitary', 'Liver', 'Kidney',
       'Cervix Uteri', 'Fallopian Tube', 'Bladder', 'Bone Marrow']


def alternet_main():

    data_path = "/data/bionets/og86asub/alternet-project/alternet/data"
    results_path = "/data/bionets/og86asub/alternet-project/alternet/raw_networks/"

    results_path_newa = "/data/bionets/og86asub/alternet-project/alternet/results_v2/"
    os.makedirs(results_path_newa, exist_ok = True)


    # Reference files
    appris_path = "appris_data.appris.txt"
    digger_path = "digger_data.csv"
    biomart_path = "biomart.txt"
    tf_list_path = "allTFs_hg38.txt"
    sf_list_path = "splicefactors.csv"

    biomart = pd.read_csv(op.join(data_path, biomart_path), sep='\t')
    tx2gene = dict(zip(biomart['Transcript stable ID'], biomart['Gene stable ID']))
    gene2tx = biomart.groupby('Gene stable ID')['Transcript stable ID'].apply(set).to_dict()
    appris_df = pd.read_csv(op.join(data_path,appris_path), sep='\t')
    digger_df = pd.read_csv(op.join(data_path,digger_path), low_memory=False)
    # Load and map TF list
    tf_list_raw = pd.read_csv(op.join(data_path,tf_list_path), sep='\t', header=None)
    tf_list = map_tf_ids(tf_list_raw, biomart)
    # Load and map SF list
    sf_list_raw = pd.read_csv(op.join(data_path, sf_list_path), header=0, sep = ',')
    sf_list = map_sf_ids(sf_list_raw.loc[:, ['Splicing_Factor']], biomart)
    # Combine TF and SF lists
    regulator_list = combine_tf_sf_lists(tf_list, sf_list)



    for TISSUE in tissues:
        print(TISSUE)
        try:
            # if op.exists(op.join(results_path_newa, f'{TISSUE}.tf.annotated.tsv')):
            #     print(f'File exisits {TISSUE}')
            #     continue
            
            transcript_data = pd.read_csv(op.join(gtex_data_dir, f'{TISSUE}.tsv'), sep = '\t', index_col = 0)


            regulator_list = pd.DataFrame({'Transcript stable ID': transcript_data['transcript_id'], 'Gene stable ID': transcript_data['gene_id']})
            tf_database = create_transcipt_annotation_database(tf_list=regulator_list, appris_df= appris_df , digger=digger_df)



            sf_net = pd.read_csv(op.join(results_path_newa,  f'{TISSUE}.sf.tsv'), sep='\t', index_col = 0)
            ambi_net = pd.read_csv(op.join(results_path_newa, f'{TISSUE}.ambi.tsv'), sep='\t', index_col = 0)
            tf_net = pd.read_csv(op.join(results_path_newa, f'{TISSUE}.tf.tsv'), sep='\t',index_col = 0)

            
            ambi_net = annotate_isoform_exclusive_edges(ambi_net, tf_database, transcript_column='source_transcript')
            ambi_net = annotate_isoform_exclusive_edges(ambi_net, tf_database, transcript_column='target_transcript', suffixes = ('_source', '_target'))
            
            sf_net = annotate_isoform_exclusive_edges(sf_net, tf_database, transcript_column='source_transcript')
            sf_net = annotate_isoform_exclusive_edges(sf_net, tf_database, transcript_column='target_transcript', suffixes = ('_source', '_target'))

            tf_net = annotate_isoform_exclusive_edges(tf_net, tf_database, transcript_column='source_transcript')
            tf_net = annotate_isoform_exclusive_edges(tf_net, tf_database, transcript_column='target_transcript', suffixes = ('_source', '_target'))



            sf_net.to_csv(op.join(results_path_newa, f'{TISSUE}.sf.annotated.tsv'), sep='\t')
            ambi_net.to_csv(op.join(results_path_newa, f'{TISSUE}.ambi.annotated.tsv'), sep='\t')
            tf_net.to_csv(op.join(results_path_newa, f'{TISSUE}.tf.annotated.tsv'), sep='\t')
        except:
            print('Tissue not found')

In [17]:
alternet_main()

Ovary


/tmp/ipykernel_3268348/2676626300.py:71: DtypeWarning: Columns (38,42,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  sf_net = pd.read_csv(op.join(results_path_newa,  f'{TISSUE}.sf.tsv'), sep='\t', index_col = 0)
/tmp/ipykernel_3268348/2676626300.py:73: DtypeWarning: Columns (11,12,29,33,34,35) have mixed types. Specify dtype option on import or set low_memory=False.
  tf_net = pd.read_csv(op.join(results_path_newa, f'{TISSUE}.tf.tsv'), sep='\t',index_col = 0)


Uterus


/tmp/ipykernel_3268348/2676626300.py:71: DtypeWarning: Columns (38,42,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  sf_net = pd.read_csv(op.join(results_path_newa,  f'{TISSUE}.sf.tsv'), sep='\t', index_col = 0)
/tmp/ipykernel_3268348/2676626300.py:73: DtypeWarning: Columns (11,12,29,33,34,35) have mixed types. Specify dtype option on import or set low_memory=False.
  tf_net = pd.read_csv(op.join(results_path_newa, f'{TISSUE}.tf.tsv'), sep='\t',index_col = 0)


Vagina


/tmp/ipykernel_3268348/2676626300.py:71: DtypeWarning: Columns (38,42,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  sf_net = pd.read_csv(op.join(results_path_newa,  f'{TISSUE}.sf.tsv'), sep='\t', index_col = 0)
/tmp/ipykernel_3268348/2676626300.py:73: DtypeWarning: Columns (11,12,29,33,34,35) have mixed types. Specify dtype option on import or set low_memory=False.
  tf_net = pd.read_csv(op.join(results_path_newa, f'{TISSUE}.tf.tsv'), sep='\t',index_col = 0)


Breast


/tmp/ipykernel_3268348/2676626300.py:71: DtypeWarning: Columns (38,42,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  sf_net = pd.read_csv(op.join(results_path_newa,  f'{TISSUE}.sf.tsv'), sep='\t', index_col = 0)
/tmp/ipykernel_3268348/2676626300.py:73: DtypeWarning: Columns (11,12,29,33,34,35) have mixed types. Specify dtype option on import or set low_memory=False.
  tf_net = pd.read_csv(op.join(results_path_newa, f'{TISSUE}.tf.tsv'), sep='\t',index_col = 0)


Skin


FileNotFoundError: [Errno 2] No such file or directory: '/data/bionets/og86asub/alternet-project/alternet/results_v2/Skin.sf.tsv'